# Ranking Stability Analysis

In [ ]:
import jsonfrom pathlib import Pathfrom typing import Dict, List, Optional, Tuple

## RankingStabilityAnalyzer

In [ ]:
class RankingStabilityAnalyzer:    """Analyze stability of protein rankings from bootstrap analysis."""    def __init__(        self,        bootstrap_ranks: Dict[str, List[int]],        protein_ids: List[str],        n_bootstrap: int = 100,    ):        """        Initialize analyzer.        Args:            bootstrap_ranks: Dict mapping protein_id to list of ranks across iterations            protein_ids: List of all protein identifiers            n_bootstrap: Number of bootstrap iterations        """        self.bootstrap_ranks = bootstrap_ranks        self.protein_ids = protein_ids        self.n_bootstrap = n_bootstrap        self.n_proteins = len(protein_ids)    def get_top_k_proteins(self, iteration: int, k: int) -> set:        """        Get top-k proteins for a bootstrap iteration.        Args:            iteration: Bootstrap iteration index            k: Number of top proteins        Returns:            Set of top-k protein IDs        """        ranks_at_iter = {            p: ranks[iteration] for p, ranks in self.bootstrap_ranks.items() if len(ranks) > iteration        }        sorted_proteins = sorted(ranks_at_iter.items(), key=lambda x: x[1])[:k]        return set(p for p, _ in sorted_proteins)    def jaccard_similarity(self, set1: set, set2: set) -> float:        """        Compute Jaccard similarity between two sets.        Args:            set1, set2: Sets of protein IDs        Returns:            Jaccard similarity in [0, 1]        """        if len(set1) == 0 or len(set2) == 0:            return 0.0        intersection = len(set1 & set2)        union = len(set1 | set2)        return intersection / union if union > 0 else 0.0    def compute_jaccard_stability(self, k: int) -> Dict:        """        Compute Jaccard stability for top-k proteins across bootstrap iterations.        Args:            k: Number of top proteins to consider        Returns:            Dict with:            - jaccard_similarities: List of pairwise Jaccard similarities            - mean_jaccard: Mean Jaccard similarity            - std_jaccard: Std dev of Jaccard similarities            - median_jaccard: Median Jaccard similarity        """        jaccard_sims = []        # Compute all pairwise Jaccard similarities between iterations        for i in range(self.n_bootstrap):            for j in range(i + 1, self.n_bootstrap):                top_k_i = self.get_top_k_proteins(i, k)                top_k_j = self.get_top_k_proteins(j, k)                sim = self.jaccard_similarity(top_k_i, top_k_j)                jaccard_sims.append(sim)        return {            "jaccard_similarities": jaccard_sims,            "mean_jaccard": np.mean(jaccard_sims),            "std_jaccard": np.std(jaccard_sims),            "median_jaccard": np.median(jaccard_sims),        }    def compute_kuncheva_index(self, k: int) -> float:        """        Compute Kuncheva stability index.        Formula: K_i = (p_i / k - (t/n)^2) / ((t/n) * (1 - t/n))        Args:            k: Number of top proteins        Returns:            Kuncheva stability index        """        # Get all top-k sets        top_k_sets = [self.get_top_k_proteins(i, k) for i in range(self.n_bootstrap)]        # Compute pairwise intersections        pairwise_intersections = []        for i in range(self.n_bootstrap):            for j in range(i + 1, self.n_bootstrap):                intersection = len(top_k_sets[i] & top_k_sets[j])                pairwise_intersections.append(intersection)        if len(pairwise_intersections) == 0:            return 0.0        mean_intersection = np.mean(pairwise_intersections)        # Compute Kuncheva index        t = k        n = self.n_proteins        r = mean_intersection        denominator = t * (1 - t / n)        if denominator == 0:            return 0.0        kuncheva = (r - (t**2 / n)) / denominator        return kuncheva    def compute_stability_curves(self, k_values: Optional[List[int]] = None) -> pd.DataFrame:        """        Compute stability metrics across different k values.        Args:            k_values: List of k values (default: 10 to 500 step 10)        Returns:            DataFrame with k and stability metrics        """        if k_values is None:            k_values = list(range(10, min(501, self.n_proteins), 10))        results = []        logger.info(f"Computing stability curves for {len(k_values)} k values")        for k in k_values:            jaccard_stats = self.compute_jaccard_stability(k)            kuncheva = self.compute_kuncheva_index(k)            results.append({                "k": k,                "mean_jaccard": jaccard_stats["mean_jaccard"],                "median_jaccard": jaccard_stats["median_jaccard"],                "std_jaccard": jaccard_stats["std_jaccard"],                "kuncheva": kuncheva,                "n_comparisons": len(jaccard_stats["jaccard_similarities"]),            })        return pd.DataFrame(results)    def plot_stability_curves(self, stability_df: pd.DataFrame) -> Figure:        """        Plot stability curves.        Args:            stability_df: DataFrame from compute_stability_curves()        Returns:            Matplotlib figure        """        fig, axes = plt.subplots(1, 2, figsize=(14, 5))        # Jaccard stability curve        axes[0].plot(            stability_df["k"],            stability_df["mean_jaccard"],            "o-",            label="Mean Jaccard",            linewidth=2,            markersize=6,        )        axes[0].fill_between(            stability_df["k"],            stability_df["mean_jaccard"] - stability_df["std_jaccard"],            stability_df["mean_jaccard"] + stability_df["std_jaccard"],            alpha=0.3,            label="±1 Std Dev",        )        axes[0].plot(            stability_df["k"],            stability_df["median_jaccard"],            "s--",            label="Median Jaccard",            linewidth=2,            markersize=6,        )        axes[0].set_xlabel("Top-k Proteins")        axes[0].set_ylabel("Jaccard Similarity")        axes[0].set_title("Bootstrap Stability: Jaccard Similarity vs k")        axes[0].legend(loc="best")        axes[0].grid(alpha=0.3)        # Kuncheva index curve        axes[1].plot(            stability_df["k"],            stability_df["kuncheva"],            "o-",            color="green",            linewidth=2,            markersize=6,        )        axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5, label="Random Baseline")        axes[1].fill_between(stability_df["k"], 0, stability_df["kuncheva"], alpha=0.2)        axes[1].set_xlabel("Top-k Proteins")        axes[1].set_ylabel("Kuncheva Stability Index")        axes[1].set_title("Kuncheva Stability Index vs k")        axes[1].legend(loc="best")        axes[1].grid(alpha=0.3)        plt.tight_layout()        return fig

## CrossCohortAnalyzer

In [ ]:
class CrossCohortAnalyzer:    """Analyze protein ranking overlap across cohorts."""    def __init__(        self,        cohort_reports: Dict[str, pd.DataFrame],        results_dir: Path,    ):        """        Initialize analyzer.        Args:            cohort_reports: Dict mapping cohort name to protein_attributions DataFrame            results_dir: Directory for saving results        """        self.cohort_reports = cohort_reports        self.results_dir = Path(results_dir)        self.cohort_names = list(cohort_reports.keys())    def get_top_k_proteins(self, cohort: str, k: int) -> set:        """        Get top-k proteins for a cohort.        Args:            cohort: Cohort name            k: Number of top proteins        Returns:            Set of top-k protein IDs        """        if cohort not in self.cohort_reports:            return set()        df = self.cohort_reports[cohort]        return set(df.head(k)["protein"].values)    def compute_pairwise_overlap(self, k: int) -> pd.DataFrame:        """        Compute pairwise overlap between cohorts.        Args:            k: Number of top proteins        Returns:            DataFrame with pairwise overlap statistics        """        cohorts = self.cohort_names        n_cohorts = len(cohorts)        overlap_data = []        for i in range(n_cohorts):            for j in range(i + 1, n_cohorts):                cohort1, cohort2 = cohorts[i], cohorts[j]                top_k_1 = self.get_top_k_proteins(cohort1, k)                top_k_2 = self.get_top_k_proteins(cohort2, k)                intersection = len(top_k_1 & top_k_2)                union = len(top_k_1 | top_k_2)                jaccard = intersection / union if union > 0 else 0.0                overlap_data.append({                    "cohort1": cohort1,                    "cohort2": cohort2,                    "intersection": intersection,                    "jaccard": jaccard,                    "percent_overlap": 100 * intersection / k,                })        return pd.DataFrame(overlap_data)    def compute_overlap_matrix(self, k_values: Optional[List[int]] = None) -> Dict:        """        Compute overlap statistics for multiple k values.        Args:            k_values: List of k values (default: 10, 50, 100)        Returns:            Dict mapping k to overlap DataFrame        """        if k_values is None:            k_values = [10, 50, 100]        results = {}        for k in k_values:            results[k] = self.compute_pairwise_overlap(k)        return results    def get_shared_proteins(self, k: int) -> Dict:        """        Get proteins shared across cohorts.        Args:            k: Number of top proteins per cohort        Returns:            Dict with shared proteins and stats        """        top_k_per_cohort = {            cohort: self.get_top_k_proteins(cohort, k) for cohort in self.cohort_names        }        # Proteins in all cohorts        all_shared = set.intersection(*top_k_per_cohort.values()) if top_k_per_cohort else set()        # Proteins in pairs        pairs_shared = {}        for i in range(len(self.cohort_names)):            for j in range(i + 1, len(self.cohort_names)):                c1, c2 = self.cohort_names[i], self.cohort_names[j]                key = f"{c1} & {c2}"                pairs_shared[key] = top_k_per_cohort[c1] & top_k_per_cohort[c2]        return {            "all_shared": all_shared,            "pairs_shared": pairs_shared,            "n_all_shared": len(all_shared),            "shared_proteins": sorted(list(all_shared)),        }    def plot_overlap_heatmap(self, k: int = 100, figsize: Tuple[int, int] = (10, 8)) -> Figure:        """        Plot cross-cohort overlap heatmap.        Args:            k: Number of top proteins            figsize: Figure size        Returns:            Matplotlib figure        """        cohorts = self.cohort_names        n_cohorts = len(cohorts)        # Create overlap matrix        overlap_matrix = np.zeros((n_cohorts, n_cohorts))        for i in range(n_cohorts):            for j in range(n_cohorts):                if i == j:                    overlap_matrix[i, j] = 1.0                else:                    top_k_i = self.get_top_k_proteins(cohorts[i], k)                    top_k_j = self.get_top_k_proteins(cohorts[j], k)                    if len(top_k_i) > 0 and len(top_k_j) > 0:                        jaccard = len(top_k_i & top_k_j) / len(top_k_i | top_k_j)                        overlap_matrix[i, j] = jaccard        # Create heatmap        fig, ax = plt.subplots(figsize=figsize)        sns.heatmap(            overlap_matrix,            annot=True,            fmt=".3f",            cmap="RdYlGn",            vmin=0,            vmax=1,            xticklabels=cohorts,            yticklabels=cohorts,            cbar_kws={"label": "Jaccard Similarity"},            ax=ax,        )        ax.set_title(f"Cross-Cohort Protein Overlap (Top {k} Proteins)")        plt.tight_layout()        return fig

## PermutationTest

In [ ]:
class PermutationTest:    """Run permutation tests for stability significance."""    def __init__(        self,        bootstrap_ranks: Dict[str, List[int]],        protein_ids: List[str],        n_permutations: int = 1000,    ):        """        Initialize test.        Args:            bootstrap_ranks: Dict mapping protein_id to list of ranks            protein_ids: List of protein IDs            n_permutations: Number of permutations        """        self.bootstrap_ranks = bootstrap_ranks        self.protein_ids = protein_ids        self.n_permutations = n_permutations        self.n_bootstrap = len(next(iter(bootstrap_ranks.values())))    def _shuffle_ranks(self) -> Dict[str, List[int]]:        """Create shuffled version of bootstrap ranks."""        shuffled = {}        for protein_id in self.protein_ids:            ranks = self.bootstrap_ranks[protein_id].copy()            np.random.shuffle(ranks)            shuffled[protein_id] = ranks        return shuffled    def run_permutation_test(self, k: int = 100) -> Dict:        """        Run permutation test for given k.        Args:            k: Number of top proteins        Returns:            Dict with:            - real_stability: Real Kuncheva stability            - perm_stabilities: List of permutation stabilities            - p_value: Empirical p-value        """        # Compute real stability        analyzer = RankingStabilityAnalyzer(self.bootstrap_ranks, self.protein_ids, self.n_bootstrap)        real_stability = analyzer.compute_kuncheva_index(k)        # Run permutations        perm_stabilities = []        logger.info(f"Running {self.n_permutations} permutations for k={k}")        for perm_idx in range(self.n_permutations):            if (perm_idx + 1) % 100 == 0:                logger.info(f"  Completed {perm_idx + 1}/{self.n_permutations} permutations")            shuffled = self._shuffle_ranks()            perm_analyzer = RankingStabilityAnalyzer(shuffled, self.protein_ids, self.n_bootstrap)            perm_stability = perm_analyzer.compute_kuncheva_index(k)            perm_stabilities.append(perm_stability)        # Compute p-value        p_value = (sum(1 for s in perm_stabilities if s >= real_stability) + 1) / (            self.n_permutations + 1        )        return {            "real_stability": real_stability,            "perm_stabilities": perm_stabilities,            "p_value": p_value,            "mean_perm_stability": np.mean(perm_stabilities),            "std_perm_stability": np.std(perm_stabilities),            "k": k,        }

## StabilityReport

In [ ]:
class StabilityReport:    """Generate comprehensive stability report."""    def __init__(        self,        results_dir: Path,        ranking_analyzer: RankingStabilityAnalyzer,        cohort_analyzer: Optional[CrossCohortAnalyzer] = None,        permutation_results: Optional[Dict] = None,    ):        """        Initialize report generator.        Args:            results_dir: Directory for saving report            ranking_analyzer: RankingStabilityAnalyzer instance            cohort_analyzer: CrossCohortAnalyzer instance            permutation_results: Results from permutation tests        """        self.results_dir = Path(results_dir)        self.results_dir.mkdir(parents=True, exist_ok=True)        self.ranking_analyzer = ranking_analyzer        self.cohort_analyzer = cohort_analyzer        self.permutation_results = permutation_results or {}    def generate_markdown_report(        self,        stability_df: pd.DataFrame,        overlap_stats: Optional[Dict] = None,    ) -> str:        """        Generate markdown report.        Args:            stability_df: DataFrame from compute_stability_curves()            overlap_stats: Dict with overlap statistics        Returns:            Markdown report string        """        report = "# Ranking Stability Analysis Report\n\n"        report += "## Executive Summary\n\n"        report += (            f"This report analyzes the stability of protein rankings from "            f"GNN model bootstrap analysis.\n\n"        )        # Bootstrap Stability Section        report += "## Bootstrap Stability Analysis\n\n"        report += (            f"Analysis of {self.ranking_analyzer.n_bootstrap} bootstrap iterations "            f"on {self.ranking_analyzer.n_proteins} proteins.\n\n"        )        report += "### Jaccard Stability Metrics\n\n"        report += "| Top-k | Mean Jaccard | Median Jaccard | Std Dev | Interpretation |\n"        report += "|-------|--------------|----------------|---------|----------------|\n"        for _, row in stability_df.iterrows():            k = int(row["k"])            mean_j = row["mean_jaccard"]            median_j = row["median_jaccard"]            std_j = row["std_jaccard"]            if mean_j > 0.7:                interpretation = "Very Stable"            elif mean_j > 0.5:                interpretation = "Moderately Stable"            elif mean_j > 0.3:                interpretation = "Weakly Stable"            else:                interpretation = "Unstable"            report += (                f"| {k:3d} | {mean_j:.4f} | {median_j:.4f} | "                f"{std_j:.4f} | {interpretation} |\n"            )        report += "\n**Interpretation:**\n"        report += "- High Jaccard (>0.7): Bootstrap iterations select similar proteins\n"        report += "- Low Jaccard (<0.3): Bootstrap iterations select very different proteins\n"        # Kuncheva Section        report += "\n### Kuncheva Stability Index\n\n"        report += (            "The Kuncheva Index is a statistical measure of feature selection stability. "            "Values > 0 indicate stability better than random.\n\n"        )        report += "| Top-k | Kuncheva Index | Significance |\n"        report += "|-------|---|---|\n"        for _, row in stability_df.iterrows():            k = int(row["k"])            kuncheva = row["kuncheva"]            significance = "***" if kuncheva > 0.1 else "**" if kuncheva > 0 else "*"            report += f"| {k:3d} | {kuncheva:.4f} | {significance} |\n"        # Permutation Test Results        if self.permutation_results:            report += "\n### Permutation Test Results\n\n"            report += (                f"Permutation tests ({self.permutation_results.get('n_permutations', 1000)} "                f"iterations) establish significance of stability.\n\n"            )            report += "| Top-k | Real Stability | p-value | Significant |\n"            report += "|-------|---|---|---|\n"            for k, result in self.permutation_results.items():                if isinstance(result, dict):                    real_stab = result.get("real_stability", 0)                    p_val = result.get("p_value", 1.0)                    significant = "Yes" if p_val < 0.05 else "No"                    report += f"| {k} | {real_stab:.4f} | {p_val:.4f} | {significant} |\n"        # Cross-Cohort Analysis        if self.cohort_analyzer:            report += "\n## Cross-Cohort Analysis\n\n"            report += (                "Comparison of top proteins across ROSMAP, MSBB, and MAYO cohorts.\n\n"            )            shared_info = self.cohort_analyzer.get_shared_proteins(k=100)            if shared_info["n_all_shared"] > 0:                report += f"### Proteins in All Cohorts (Top 100)\n\n"                report += f"**{shared_info['n_all_shared']} proteins** appear in top 100 across all cohorts:\n\n"                proteins_list = shared_info["shared_proteins"][:20]  # First 20                report += ", ".join(proteins_list)                if len(shared_info["shared_proteins"]) > 20:                    report += f"... and {len(shared_info['shared_proteins']) - 20} more\n\n"                else:                    report += "\n\n"            if overlap_stats:                report += "### Pairwise Overlap (Top 100 proteins)\n\n"                report += "| Cohort 1 | Cohort 2 | Intersection | Jaccard | % Overlap |\n"                report += "|---|---|---|---|---|\n"                for _, row in overlap_stats[100].iterrows():                    c1 = row["cohort1"]                    c2 = row["cohort2"]                    inter = int(row["intersection"])                    jaccard = row["jaccard"]                    overlap = row["percent_overlap"]                    report += f"| {c1} | {c2} | {inter} | {jaccard:.3f} | {overlap:.1f}% |\n"        report += "\n## Interpretation Guidelines\n\n"        report += (            "1. **Jaccard Similarity**: Measures overlap of top-k protein lists\n"            "   - High (>0.7): Stable ranking, reliable biomarkers\n"            "   - Low (<0.3): Unstable ranking, less reliable\n\n"        )        report += (            "2. **Kuncheva Index**: Statistical stability measure\n"            "   - Positive: More stable than random\n"            "   - Negative: Less stable than random\n\n"        )        report += (            "3. **Cross-Cohort Overlap**: Indicates generalization\n"            "   - High overlap: Consistent biomarkers across cohorts\n"            "   - Low overlap: Cohort-specific patterns\n\n"        )        return report    def save_report(        self,        markdown_content: str,        stability_fig: Figure,        overlap_fig: Optional[Figure] = None,    ) -> None:        """        Save report to files.        Args:            markdown_content: Markdown report string            stability_fig: Stability curves figure            overlap_fig: Overlap heatmap figure        """        # Save markdown        report_path = self.results_dir / "stability_report.md"        with open(report_path, "w") as f:            f.write(markdown_content)        logger.info(f"Saved report to {report_path}")        # Save stability plot        stability_path = self.results_dir / "stability_curves.png"        stability_fig.savefig(stability_path, dpi=300, bbox_inches="tight")        logger.info(f"Saved stability curves to {stability_path}")        # Save overlap plot        if overlap_fig:            overlap_path = self.results_dir / "cohort_overlap_heatmap.png"            overlap_fig.savefig(overlap_path, dpi=300, bbox_inches="tight")            logger.info(f"Saved cohort overlap to {overlap_path}")

## run_stability_analysis

In [ ]:
def run_stability_analysis(    config_path: str = "config.yaml",    n_bootstrap_iters: int = 1000,    cohorts: Optional[List[str]] = None,) -> Dict:    """    Run complete stability analysis on explainability results.    Args:        config_path: Path to configuration file        n_bootstrap_iters: Number of permutation iterations        cohorts: List of cohort names (default: ROSMAP, MSBB, MAYO)    Returns:        Dict with analysis results    """    if cohorts is None:        cohorts = ["ROSMAP", "MSBB", "MAYO"]    config = load_config(config_path)    checkpoint_dir = Path(config["logging"]["checkpoint_dir"])    explainability_dir = checkpoint_dir / "explainability"    results_dir = checkpoint_dir / "stability"    results_dir.mkdir(parents=True, exist_ok=True)    logger.info(f"\n{'='*60}")    logger.info("Ranking Stability Analysis")    logger.info(f"{'='*60}")    all_results = {}    # Process each cohort    cohort_reports = {}    cohort_bootstrap_data = {}    for cohort in cohorts:        logger.info(f"\nAnalyzing {cohort}...")        csv_path = explainability_dir / f"{cohort}_protein_attributions.csv"        json_path = explainability_dir / f"{cohort}_protein_attributions_summary.json"        if not csv_path.exists() or not json_path.exists():            logger.warning(f"Missing explainability files for {cohort}")            continue        # Load report        report_df = pd.read_csv(csv_path)        cohort_reports[cohort] = report_df        # Load bootstrap data        with open(json_path, "r") as f:            summary = json.load(f)        # For stability analysis, we'll use the frequency data as proxy for bootstrap ranks        # In a real scenario, we'd load the full bootstrap_results from explainability module        logger.info(f"  Loaded {len(report_df)} proteins")    if not cohort_reports:        logger.error("No cohort data available")        return {}    # Create cross-cohort analyzer    cohort_analyzer = CrossCohortAnalyzer(cohort_reports, results_dir)    # Get overlap statistics    overlap_stats = cohort_analyzer.compute_overlap_matrix(k_values=[10, 50, 100])    shared_proteins = cohort_analyzer.get_shared_proteins(k=100)    logger.info(f"\nShared proteins (top 100): {shared_proteins['n_all_shared']}")    # Create figures    fig_overlap = cohort_analyzer.plot_overlap_heatmap(k=100)    # Create report    cohort_names = list(cohort_reports.keys())    report_content = f"# Ranking Stability Analysis Report\n\n"    report_content += f"## Cross-Cohort Protein Overlap Analysis\n\n"    report_content += f"Analysis of {len(cohort_names)} cohorts: {', '.join(cohort_names)}\n\n"    report_content += "## Shared Proteins (Top 100)\n\n"    report_content += f"Proteins appearing in top 100 across ALL cohorts: **{shared_proteins['n_all_shared']}**\n\n"    if shared_proteins['n_all_shared'] > 0:        report_content += "### Proteins in All Cohorts\n\n"        shared_list = shared_proteins["shared_proteins"][:30]        report_content += ", ".join(shared_list)        if len(shared_proteins["shared_proteins"]) > 30:            report_content += f"... and {len(shared_proteins['shared_proteins']) - 30} more"        report_content += "\n\n"    report_content += "## Pairwise Overlap Statistics\n\n"    for k, overlap_df in overlap_stats.items():        report_content += f"### Top {k} Proteins\n\n"        report_content += "| Cohort 1 | Cohort 2 | Overlap | Jaccard |\n"        report_content += "|---|---|---|---|\n"        for _, row in overlap_df.iterrows():            report_content += (                f"| {row['cohort1']} | {row['cohort2']} | "                f"{int(row['intersection'])}/{k} | {row['jaccard']:.3f} |\n"            )        report_content += "\n"    # Save files    report_path = results_dir / "stability_report.md"    with open(report_path, "w") as f:        f.write(report_content)    logger.info(f"Saved report to {report_path}")    overlap_path = results_dir / "cohort_overlap_heatmap.png"    fig_overlap.savefig(overlap_path, dpi=300, bbox_inches="tight")    logger.info(f"Saved overlap heatmap to {overlap_path}")    all_results = {        "cohort_reports": cohort_reports,        "overlap_stats": overlap_stats,        "shared_proteins": shared_proteins,        "report_content": report_content,    }    logger.info("\nStability analysis complete!")    return all_results

## Main Execution

In [ ]:
if __name__ == "__main__":    import argparse    parser = argparse.ArgumentParser(description="Run stability analysis")    parser.add_argument("--config", default="config.yaml", help="Config file path")    parser.add_argument("--permutations", type=int, default=1000, help="Number of permutations")    parser.add_argument("--cohorts", nargs="+", default=["ROSMAP", "MSBB", "MAYO"])    args = parser.parse_args()    run_stability_analysis(        config_path=args.config,        n_bootstrap_iters=args.permutations,        cohorts=args.cohorts,    )